### General Imports:


In [ ]:
import os
import gymnasium as gym
from stable_baselines3.ppo import PPO
from stable_baselines3.ppo.policies import MlpPolicy as MLP_PPO
from netsim.netSimPy import *
from netsim.gym_basic.envs import RMSA_ENV
from stable_baselines3.common.monitor import Monitor
import numpy as np
import tensorflow as tf
import logging
from IPython.display import clear_output

logging.getLogger("tensorflow").setLevel(logging.FATAL)
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import sync_envs_normalization

# from netsim.allocators import sap_ff, ldpb, sap_ff2
from stable_baselines3.common.evaluation import evaluate_policy
from netsim.netSimPy.common.allocators import sap_ff
from netsim.netSimPy.common.utils import optimize, sample_PPO_params

tf.get_logger().setLevel("INFO")
tf.__version__

/Users/jbcedeno/Documents/projcts/multiband-gymnasium/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'2.16.2'

In [ ]:
def warming_lr(initial_lr=1.5e-4, max_warm_lr=3e-4, final_lr=2e-4, warmup_progress=0.85):
    def learning_rate_fn(progress):
        if progress >= warmup_progress:
            m = (initial_lr - max_warm_lr) / (1 - warmup_progress)
            n = initial_lr - m
            return round(m * progress + n, 6)
        else:
            m = (max_warm_lr - final_lr) / warmup_progress
            return round(m * progress + final_lr, 6)

    return learning_rate_fn

In [ ]:
from typing import Union, Optional


class MyBestCallback(EvalCallback):
    def __init__(
        self,
        eval_env,
        callback_on_new_best=None,
        callback_after_eval=None,
        n_eval_episodes: int = 5,
        eval_freq: int = 10000,
        log_path: Optional[str] = None,
        best_model_save_path: Optional[str] = None,
        deterministic: bool = True,
        render: bool = False,
        verbose: int = 1,
        warn: bool = True,
    ):
        super().__init__(
            eval_env,
            callback_on_new_best,
            callback_after_eval,
            n_eval_episodes,
            eval_freq,
            log_path,
            best_model_save_path,
            deterministic,
            render,
            verbose,
            warn,
        )
        self.eval_env = eval_env
        self.best_reward_at = 0

    def _on_step(self) -> bool:
        continue_training = True

        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            # Sync training and eval env if there is VecNormalize
            if self.model.get_vec_normalize_env() is not None:
                try:
                    sync_envs_normalization(self.training_env, self.eval_env)
                except AttributeError as e:
                    raise AssertionError(
                        "Training and eval env are not wrapped the same way, "
                        "see https://stable-baselines3.readthedocs.io/en/master/guide/callbacks.html#evalcallback "
                        "and warning above."
                    ) from e

            # Reset success rate buffer
            self._is_success_buffer = []
            # self.eval_env.reset(hard_reset=True)
            episode_rewards, episode_lengths = evaluate_policy(
                self.model,
                self.eval_env,
                n_eval_episodes=self.n_eval_episodes,
                render=self.render,
                deterministic=self.deterministic,
                return_episode_rewards=True,
                warn=self.warn,
                callback=self._log_success_callback,
            )

            if self.log_path is not None:
                self.evaluations_timesteps.append(self.num_timesteps)
                self.evaluations_results.append(episode_rewards)
                self.evaluations_length.append(episode_lengths)

                kwargs = {}
                # Save success log if present
                if len(self._is_success_buffer) > 0:
                    self.evaluations_successes.append(self._is_success_buffer)
                    kwargs = dict(successes=self.evaluations_successes)

                np.savez(
                    self.log_path,
                    timesteps=self.evaluations_timesteps,
                    results=self.evaluations_results,
                    ep_lengths=self.evaluations_length,
                    **kwargs,
                )

            mean_reward, std_reward = np.mean(episode_rewards), np.std(episode_rewards)
            mean_ep_length, std_ep_length = np.mean(episode_lengths), np.std(
                episode_lengths
            )
            self.last_mean_reward = mean_reward

            if self.verbose >= 1:
                print(
                    f"Eval num_timesteps={self.num_timesteps}, "
                    f"episode_reward={mean_reward:.2f} +/- {std_reward:.2f}"
                )
            # Add to current Logger
            self.logger.record("eval/mean_reward", float(mean_reward))
            self.logger.record("eval/mean_ep_length", mean_ep_length)

            if len(self._is_success_buffer) > 0:
                success_rate = np.mean(self._is_success_buffer)
                if self.verbose >= 1:
                    print(f"Success rate: {100 * success_rate:.2f}%")
                self.logger.record("eval/success_rate", success_rate)

            # Dump log so the evaluation results are printed with the correct timestep
            self.logger.record(
                "time/total_timesteps", self.num_timesteps, exclude="tensorboard"
            )
            self.logger.dump(self.num_timesteps)

            if mean_reward >= self.best_mean_reward:
                self.best_reward_at = self.num_timesteps
                if self.best_model_save_path is not None:
                    self.model.save(os.path.join(self.best_model_save_path, "best_model"))
                self.best_mean_reward = mean_reward
                # Trigger callback on new best model, if needed
                if self.callback_on_new_best is not None:
                    continue_training = self.callback_on_new_best.on_step()
            if self.verbose >= 1:
                print(
                    f"Best mean reward: {self.best_mean_reward:.2f} at timestep No. {self.best_reward_at}"
                )
                clear_output(wait=True)
            # Trigger callback after every evaluation, if needed
            if self.callback is not None:
                continue_training = continue_training and self._on_event()

        return continue_training

### Training:


In [ ]:
from netsim.netSimPy.common.allocators import sap_ff

__file__ = "RMSA.ipynb"
absolutepath = os.path.abspath(__file__)
file_name = os.path.basename(os.path.abspath(__file__)).split(".")[0]
networkPaths = "/Users/jbcedeno/Documents/projcts/multiband-gymnasium/networks/nsfnet"

log_dir = f"./tmp/{file_name}/"
os.makedirs(log_dir, exist_ok=True)
tensorboard_log = f"./tb/{file_name}/nsfnet/"

M_LAMBDA = 150000
N_EVALUATIONS_EPISODES = 100
EPISODE_LENGTH = 1000


def objetive(trial):
    network = Network(
        networkFileName=networkPaths + "/network.json",
        pathsFileName=networkPaths + "/routes.json",
        bitrateFilename=networkPaths + "/bitrates_c_bands.json",
    )
    generator = EventsGenerator(mLambda=M_LAMBDA)

    sim_args = dict(
        network=network,
        eventsGenerator=generator,
    )
    simulator = NetworkSimulator(**sim_args)

    env_args = dict(
        simulator=simulator,
        episode_length=EPISODE_LENGTH,
        j=5,
        n_paths=3,
    )
    env = Monitor(gym.make("RMSA_ENV-v0", **env_args), log_dir)

    mArgs = sample_PPO_params(trial)
    model = PPO(MLP_PPO, env, verbose=0, seed=3, tensorboard_log=tensorboard_log, **mArgs)

    model.learn(total_timesteps=150000)
    mean_reward, _ = evaluate_policy(
        model, env, n_eval_episodes=N_EVALUATIONS_EPISODES, deterministic=True
    )
    return mean_reward

In [5]:
optimize(objetive, 50, 2)

[I 2025-03-11 09:01:34,541] A new study created in memory with name: NoName
[I 2025-03-11 09:05:48,227] Trial 0 finished with value: 622.64 and parameters: {'learning_rate': 0.001, 'gamma': 0.9500000000000001, 'n_steps': 9, 'n_epochs': 25}. Best is trial 0 with value: 622.64.
[I 2025-03-11 09:09:27,332] Trial 1 finished with value: 641.06 and parameters: {'learning_rate': 0.0007000000000000001, 'gamma': 0.93, 'n_steps': 9, 'n_epochs': 10}. Best is trial 1 with value: 641.06.
[I 2025-03-11 09:13:08,921] Trial 2 finished with value: 678.68 and parameters: {'learning_rate': 0.00030000000000000003, 'gamma': 0.92, 'n_steps': 8, 'n_epochs': 10}. Best is trial 2 with value: 678.68.
[I 2025-03-11 09:16:58,226] Trial 3 finished with value: 635.86 and parameters: {'learning_rate': 0.0001, 'gamma': 0.92, 'n_steps': 6, 'n_epochs': 13}. Best is trial 2 with value: 678.68.
[I 2025-03-11 09:21:03,166] Trial 4 finished with value: 668.34 and parameters: {'learning_rate': 0.0002, 'gamma': 0.98, 'n_step

Number of finished trials:  50
Best trial:
  Value:  680.96
  Params: 
    learning_rate: 0.0009000000000000001
    gamma: 0.9500000000000001
    n_steps: 6
    n_epochs: 16
  User attrs:
    gamma: 0.95
    learning_rate: 0.0009000000000000001
    n_steps: 64
    n_epochs: 16


In [6]:
# best_model = PPO.load(f"./tmp/{file_name}/best_model.zip")
# mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=10, deterministic=True)
# print(mean_reward, std_reward)

In [7]:
# train_model = PPO.load(f"./tmp/{file_name}/best_model.zip")
# env.setLambda(100000)

In [ ]:
# mean_reward, std_reward = evaluate_policy(train_model, env, n_eval_episodes=100, deterministic=True)
# print(mean_reward)

In [9]:
# env.setAllocatorFunc(sap_ff2(3))
# mean_reward, std_reward = evaluate_policy(train_model, env, n_eval_episodes=100, deterministic=True)
# print(mean_reward)

In [ ]:
import math

x = [1, 4, 7, 11, 13, 18, 24, 50, 61, 67, 90, 92]


def breadSpectrum(valList: list, n):
    spectrum_obs = [0] * n
    i = 0
    ii = len(valList) - 1
    counter = 0
    while i <= ii:
        print(i, ii, counter)
        if spectrum_obs[counter] == 0:
            spectrum_obs[counter] = valList[i]
        else:
            break
        if spectrum_obs[len(spectrum_obs) - counter - 1] == 0:
            spectrum_obs[len(spectrum_obs) - counter - 1] = valList[ii]
        else:
            break
        i += 1
        ii -= 1
        counter += 1

    return spectrum_obs


breadSpectrum(x, 13)

0 11 0
1 10 1
2 9 2
3 8 3
4 7 4
5 6 5


[1, 4, 7, 11, 13, 18, 0, 24, 50, 61, 67, 90, 92]

In [ ]:
def breadSpectrum(val_list: list, n: int):
    spectrum_obs = [0] * n
    start, end = 0, len(val_list) - 1
    # Fill the spectrum from both ends
    for counter in range(n // 2 + (n % 2)):
        if start <= end:
            spectrum_obs[counter] = val_list[start]  # Assign from the start
            start += 1
        if start <= end and spectrum_obs[-(counter + 1)] == 0:
            spectrum_obs[-(counter + 1)] = val_list[end]  # Assign from the end
            end -= 1

    return spectrum_obs


# Example usage
x = [3, 9, 16, 19, 29, 32, 84]
result = breadSpectrum(x, 14)
print("Spectrum Observation:", result)

Spectrum Observation: [3, 9, 16, 19, 0, 0, 0, 0, 0, 0, 0, 29, 32, 84]


In [ ]:
def compute_mask(env):
    n_routes = 3
    n_blocks = 5
    start = 28  # n_nodes * 2
    n_feautures = n_blocks * 2 + 3

    def _mask(sample):
        mask = [0] * n_routes * n_blocks
        for idp in range(n_routes):
            for idb in range(n_blocks):
                pos = start + idp * (n_feautures) + idb * 2
                if sample[pos] != -1:
                    mask[idp * n_blocks + idb] = 1
        return mask

    return _mask

In [20]:
sample = [-1] * 67
sample[28] = 4
sample[29] = 15
sample[41] = 2

In [21]:
masking = compute_mask(2)
masking(sample)

{28}
{30}
{32}
{34}
{36}
{41}
{43}
{45}
{47}
{49}
{54}
{56}
{58}
{60}
{62}


[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]